In [1]:
import pandas as pd


df = pd.DataFrame({
    "order_id": [
        1001, 1002, 1003, 1004, 1005,
        1006, 1007, 1008, 1009, 1010
    ],

    "member_level": [
        3, 1, 2, 3, 2,
        1, 3, 2, 1, 3
    ],

    "category": [
        "Electronics",
        "Books",
        "Clothing",
        "Electronics",
        "Books",
        "Clothing",
        "Electronics",
        "Books",
        "Electronics",
        "Clothing"
    ],

    "price": [
        1200, 80, 350, 600, 120,
        450, 1500, 90, 700, 300
    ],

    "quantity": [
        1, 4, 2, 2, 2,
        1, 1, 5, 1, 3
    ],

    "refunded": [
        False, False, True, False, False,
        False, True, False, False, False
    ]
})

df

,order_id,member_level,category,price,quantity,refunded
0,1001,3,Electronics,1200,1,False
1,1002,1,Books,80,4,False
2,1003,2,Clothing,350,2,True
3,1004,3,Electronics,600,2,False
4,1005,2,Books,120,2,False
5,1006,1,Clothing,450,1,False
6,1007,3,Electronics,1500,1,True
7,1008,2,Books,90,5,False
8,1009,1,Electronics,700,1,False
9,1010,3,Clothing,300,3,False


In [2]:
df_cleaned = df.copy()

## 任务 1：计算订单金额

新增：

```python
amount
```

计算规则：

`amount = price × quantity`

In [3]:
df_cleaned = (
    df
    .assign(
        amount = lambda x:x['price'] * x['quantity']
    )
)
df_cleaned

,order_id,member_level,category,price,quantity,refunded,amount
0,1001,3,Electronics,1200,1,False,1200
1,1002,1,Books,80,4,False,320
2,1003,2,Clothing,350,2,True,700
3,1004,3,Electronics,600,2,False,1200
4,1005,2,Books,120,2,False,240
5,1006,1,Clothing,450,1,False,450
6,1007,3,Electronics,1500,1,True,1500
7,1008,2,Books,90,5,False,450
8,1009,1,Electronics,700,1,False,700
9,1010,3,Clothing,300,3,False,900


## 任务 2：生成订单金额等级

新增：

```python
amount_level
```

业务规则：

- amount >= 1000 → `"HIGH"`
- 500 <= amount < 1000 → `"MEDIUM"`
- amount < 500 → `"LOW"`


In [4]:
import numpy as np
df_cleaned = (
    df_cleaned
    .assign(
        amount_level = lambda x:np.select(
            [
                x['amount'] >= 1000,
                x['amount'] >= 500
            ],
            [
                'HIGH',
                'MEDIUM'
            ],
            default = 'LOW'
        )
    )
)
df_cleaned

,order_id,member_level,category,price,quantity,refunded,amount,amount_level
0,1001,3,Electronics,1200,1,False,1200,HIGH
1,1002,1,Books,80,4,False,320,LOW
2,1003,2,Clothing,350,2,True,700,MEDIUM
3,1004,3,Electronics,600,2,False,1200,HIGH
4,1005,2,Books,120,2,False,240,LOW
5,1006,1,Clothing,450,1,False,450,LOW
6,1007,3,Electronics,1500,1,True,1500,HIGH
7,1008,2,Books,90,5,False,450,LOW
8,1009,1,Electronics,700,1,False,700,MEDIUM
9,1010,3,Clothing,300,3,False,900,MEDIUM


## 任务 3：转换用户等级

当前：

```text
member_level
1
2
3
```

业务含义：

```text
1 → NORMAL
2 → SILVER
3 → GOLD
```

新增：

```python
member_name
```

要求：

不要修改原来的 `member_level`。

In [6]:
level_map = {
    1:'NORMAL',
    2:'SILVER',
    3:'GOLD'
}
df_cleaned = (
    df_cleaned
    .assign(
        member_name = lambda x:x['member_level'].map(level_map)
    )
)
df_cleaned

,order_id,member_level,category,price,quantity,refunded,amount,amount_level,member_name
0,1001,3,Electronics,1200,1,False,1200,HIGH,GOLD
1,1002,1,Books,80,4,False,320,LOW,NORMAL
2,1003,2,Clothing,350,2,True,700,MEDIUM,SILVER
3,1004,3,Electronics,600,2,False,1200,HIGH,GOLD
4,1005,2,Books,120,2,False,240,LOW,SILVER
5,1006,1,Clothing,450,1,False,450,LOW,NORMAL
6,1007,3,Electronics,1500,1,True,1500,HIGH,GOLD
7,1008,2,Books,90,5,False,450,LOW,SILVER
8,1009,1,Electronics,700,1,False,700,MEDIUM,NORMAL
9,1010,3,Clothing,300,3,False,900,MEDIUM,GOLD


## 任务 4：计算实际销售额

新增：

```python
net_amount
```

业务规则：

如果订单已经退款：

```text
net_amount = 0
```

否则：

```text
net_amount = amount
```

In [14]:
df_cleaned = (
    df_cleaned
    .assign(
        net_amount = lambda x:np.where(
            x['refunded'],
            0,
            x['amount']
        )
    )
)
df_cleaned

,order_id,member_level,category,price,quantity,refunded,amount,amount_level,member_name,net_amount,order_tag
0,1001,3,Electronics,1200,1,False,1200,HIGH,GOLD,1200,HIGH_VALUE_ELECTRONICS
1,1002,1,Books,80,4,False,320,LOW,NORMAL,320,BULK_BOOK_ORDER
2,1003,2,Clothing,350,2,True,700,MEDIUM,SILVER,0,NORMAL_ORDER
3,1004,3,Electronics,600,2,False,1200,HIGH,GOLD,1200,HIGH_VALUE_ELECTRONICS
4,1005,2,Books,120,2,False,240,LOW,SILVER,240,NORMAL_ORDER
5,1006,1,Clothing,450,1,False,450,LOW,NORMAL,450,NORMAL_ORDER
6,1007,3,Electronics,1500,1,True,1500,HIGH,GOLD,0,HIGH_VALUE_ELECTRONICS
7,1008,2,Books,90,5,False,450,LOW,SILVER,450,BULK_BOOK_ORDER
8,1009,1,Electronics,700,1,False,700,MEDIUM,NORMAL,700,NORMAL_ORDER
9,1010,3,Clothing,300,3,False,900,MEDIUM,GOLD,900,NORMAL_ORDER


## 任务 5：生成商品业务标签

业务部门需要根据商品类别和订单金额生成：

```python
order_tag
```

规则如下：

### Electronics

如果：

```text
category == "Electronics"
并且
amount >= 1000
```

标记：

```text
HIGH_VALUE_ELECTRONICS
```

### Books

如果：

```text
category == "Books"
并且
quantity >= 3
```

标记：

```text
BULK_BOOK_ORDER
```

### 其它情况

统一标记：

```text
NORMAL_ORDER
```

In [12]:
df_cleaned = (
    df_cleaned
    .assign(
        order_tag = lambda x:np.select(
            [
            (x['category'] == "Electronics" ) & (x['amount'] >= 1000),
            (x['category'] == "Books") & (x['quantity'] >= 3)
            ],
            [
                'HIGH_VALUE_ELECTRONICS',
                'BULK_BOOK_ORDER'
            ],
            default = 'NORMAL_ORDER'
        )
    )
)
df_cleaned

,order_id,member_level,category,price,quantity,refunded,amount,amount_level,member_name,net_amount,order_tag
0,1001,3,Electronics,1200,1,False,1200,HIGH,GOLD,1200,HIGH_VALUE_ELECTRONICS
1,1002,1,Books,80,4,False,320,LOW,NORMAL,320,BULK_BOOK_ORDER
2,1003,2,Clothing,350,2,True,700,MEDIUM,SILVER,0,NORMAL_ORDER
3,1004,3,Electronics,600,2,False,1200,HIGH,GOLD,1200,HIGH_VALUE_ELECTRONICS
4,1005,2,Books,120,2,False,240,LOW,SILVER,240,NORMAL_ORDER
5,1006,1,Clothing,450,1,False,450,LOW,NORMAL,450,NORMAL_ORDER
6,1007,3,Electronics,1500,1,True,1500,HIGH,GOLD,0,HIGH_VALUE_ELECTRONICS
7,1008,2,Books,90,5,False,450,LOW,SILVER,450,BULK_BOOK_ORDER
8,1009,1,Electronics,700,1,False,700,MEDIUM,NORMAL,700,NORMAL_ORDER
9,1010,3,Clothing,300,3,False,900,MEDIUM,GOLD,900,NORMAL_ORDER


## 任务 6：生成最终分析表

最终保留：

- order_id
- member_level
- member_name
- category
- amount
- amount_level
- refunded
- net_amount
- order_tag

保存为：

```python
result
```

不要修改原始 `df`。

In [13]:
result = (
    df_cleaned
    [
        [
            'order_id',
            'member_level',
            'member_name',
            'category',
            'amount',
            'amount_level',
            'refunded',
            'net_amount',
            'order_tag'
            
        ]
    ]
)
result

,order_id,member_level,member_name,category,amount,amount_level,refunded,net_amount,order_tag
0,1001,3,GOLD,Electronics,1200,HIGH,False,1200,HIGH_VALUE_ELECTRONICS
1,1002,1,NORMAL,Books,320,LOW,False,320,BULK_BOOK_ORDER
2,1003,2,SILVER,Clothing,700,MEDIUM,True,0,NORMAL_ORDER
3,1004,3,GOLD,Electronics,1200,HIGH,False,1200,HIGH_VALUE_ELECTRONICS
4,1005,2,SILVER,Books,240,LOW,False,240,NORMAL_ORDER
5,1006,1,NORMAL,Clothing,450,LOW,False,450,NORMAL_ORDER
6,1007,3,GOLD,Electronics,1500,HIGH,True,0,HIGH_VALUE_ELECTRONICS
7,1008,2,SILVER,Books,450,LOW,False,450,BULK_BOOK_ORDER
8,1009,1,NORMAL,Electronics,700,MEDIUM,False,700,NORMAL_ORDER
9,1010,3,GOLD,Clothing,900,MEDIUM,False,900,NORMAL_ORDER
